# Argo Float Finder

Discovers which Argo floats operated inside the CMEMS model domain used by this project.

**What it does:**
1. Reads `data/raw/manifest.yaml` to determine the full spatial and temporal extent of the downloaded CMEMS tiles.
2. Queries the live Argo index for every float that profiled in that region during that time window.
3. Shows a summary table and an interactive map of all matching float tracks.

**Then:** Edit the `WMOS_TO_DOWNLOAD` list and run the download cell to fetch profile data for any float of interest.

**Run all cells:** `Kernel › Restart & Run All`

In [ ]:
import sys
from pathlib import Path
from datetime import datetime

import pandas as pd
import plotly.graph_objects as go

sys.path.insert(0, str(Path("..") / "src"))
from data_loader import load_manifest

DATA_DIR = Path("..") / "data" / "raw"

# ── 1. Derive CMEMS domain from manifest ─────────────────────────────────────
manifest = load_manifest(DATA_DIR)

lat_min = min(e["lat"][0] for e in manifest)
lat_max = max(e["lat"][1] for e in manifest)
lon_min = min(e["lon"][0] for e in manifest)
lon_max = max(e["lon"][1] for e in manifest)
t_start = min(datetime.fromisoformat(e["time"][0]) for e in manifest)
t_end   = max(datetime.fromisoformat(e["time"][1]) for e in manifest)

print(f"CMEMS domain : lat=[{lat_min:.2f}, {lat_max:.2f}]  lon=[{lon_min:.2f}, {lon_max:.2f}]")
print(f"Time window  : {t_start.date()} → {t_end.date()}")
print(f"Tiles loaded : {len(manifest)}")

# ── 2. Load Argo global index and filter to CMEMS domain ─────────────────────
from argopy import ArgoIndex

print("\nLoading Argo global index...")
df_all = ArgoIndex().load().to_dataframe()  # wmo, dac, latitude, longitude, date columns

mask = (
    (df_all["latitude"]  >= lat_min) & (df_all["latitude"]  <= lat_max) &
    (df_all["longitude"] >= lon_min) & (df_all["longitude"] <= lon_max) &
    (df_all["date"] >= pd.Timestamp(t_start)) &
    (df_all["date"] <= pd.Timestamp(t_end))
)
df = df_all[mask].copy()

# ── 3. Summarise by WMO ───────────────────────────────────────────────────────
summary = (
    df.groupby("wmo")
    .agg(
        n_profiles=("date", "count"),
        first_profile=("date", "min"),
        last_profile=("date", "max"),
    )
    .sort_values("n_profiles", ascending=False)
    .reset_index()
)

print(f"\nFound {len(summary)} floats with {len(df)} total profiles in the domain:")
print(summary.to_string(index=False))

# ── 4. Map: domain box + each float's track ───────────────────────────────────
fig = go.Figure()

fig.add_trace(go.Scattergeo(
    lon=[lon_min, lon_max, lon_max, lon_min, lon_min],
    lat=[lat_min, lat_min, lat_max, lat_max, lat_min],
    mode="lines",
    line=dict(color="cyan", width=2, dash="dash"),
    name="CMEMS domain",
    showlegend=True,
))

for wmo in summary["wmo"]:
    pts = df[df["wmo"] == wmo].sort_values("date")
    fig.add_trace(go.Scattergeo(
        lon=pts["longitude"],
        lat=pts["latitude"],
        mode="lines+markers",
        marker=dict(size=3),
        line=dict(width=1),
        name=str(wmo),
        hovertemplate=(
            f"WMO {wmo}<br>"
            "%{lat:.3f}°N  %{lon:.3f}°E<br>"
            "<extra></extra>"
        ),
    ))

fig.update_layout(
    title=dict(
        text=(
            f"Argo floats in CMEMS domain — {len(summary)} floats, {len(df)} profiles<br>"
            f"<sup>{t_start.date()} → {t_end.date()}</sup>"
        ),
        x=0.5, xanchor="center",
    ),
    geo=dict(
        scope="europe",
        center=dict(lat=(lat_min + lat_max) / 2, lon=(lon_min + lon_max) / 2),
        projection_scale=6,
        showland=True,  landcolor="rgb(40,40,40)",
        showocean=True, oceancolor="rgb(10,30,60)",
        showcoastlines=True, coastlinecolor="grey",
        showframe=False,
    ),
    paper_bgcolor="rgb(15,20,40)",
    font_color="white",
    height=650,
    legend=dict(title="WMO", bgcolor="rgba(0,0,0,0.5)"),
)
fig.show()

## Download float profiles

Add WMO numbers from the map above to `WMOS_TO_DOWNLOAD`, then run the next cell.
Profiles are saved to `data/argo_data/argo_{wmo}/{wmo}_profiles.nc` using the same format
as the existing floats in this project. Already-downloaded floats are skipped automatically.

In [ ]:
# ── Edit this list ─────────────────────────────────────────────────────────────
WMOS_TO_DOWNLOAD = [
    # e.g. 7902194, 4903784
]

In [ ]:
from argopy import DataFetcher as ArgoDataFetcher

ARGO_DATA_DIR = Path("..") / "data" / "argo_data"

if not WMOS_TO_DOWNLOAD:
    print("WMOS_TO_DOWNLOAD is empty — add WMO numbers to the cell above.")
else:
    for wmo in WMOS_TO_DOWNLOAD:
        out_dir  = ARGO_DATA_DIR / f"argo_{wmo}"
        out_path = out_dir / f"{wmo}_profiles.nc"

        if out_path.exists():
            print(f"WMO {wmo}: already present at {out_path}, skipping.")
            continue

        print(f"WMO {wmo}: downloading from ERDDAP...")
        out_dir.mkdir(parents=True, exist_ok=True)

        loader = ArgoDataFetcher(src="erddap").float(wmo)
        ds = loader.to_xarray()
        ds = ds.argo.point2profile()
        ds.to_netcdf(out_path)
        print(f"WMO {wmo}: saved {int(ds.dims['N_PROF'])} profiles → {out_path}")

    print("\nDone.")